## K-En Yakın Komşu (Model)

*Bir önceki bölümde kullandığımız Hitters veri setini kullanacağız.*

In [1]:
import numpy as np
import pandas as pd 
from sklearn.model_selection import train_test_split, GridSearchCV,cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
from sklearn.preprocessing import scale 
from sklearn import model_selection
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import BaggingRegressor
import os
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["LOKY_MAX_CPU_COUNT"] = "4"  
from warnings import filterwarnings
filterwarnings('ignore')

In [2]:
#veri seti yukleme, hazirlama ve on isleme

In [3]:
hit = pd.read_csv("hitters.csv")
df = hit.copy()
df = df.dropna()
dms = pd.get_dummies(df[["League", "Division", "NewLeague"]])
dms = dms.astype(int)
y = df["Salary"]
X_ = df.drop(["Salary", "League", "Division", "NewLeague"], axis = 1).astype("float64")
X = pd.concat([X_, dms[["League_N", "Division_W", "NewLeague_N"]]], axis = 1)
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y,
                                                    test_size = 0.25,
                                                    random_state = 42)

In [4]:
X.head()

,AtBat,Hits,HmRun,Runs,RBI,Walks,Years,CAtBat,CHits,CHmRun,CRuns,CRBI,CWalks,PutOuts,Assists,Errors,League_N,Division_W,NewLeague_N
1,315.0,81.0,7.0,24.0,38.0,39.0,14.0,3449.0,835.0,69.0,321.0,414.0,375.0,632.0,43.0,10.0,1,1,1
2,479.0,130.0,18.0,66.0,72.0,76.0,3.0,1624.0,457.0,63.0,224.0,266.0,263.0,880.0,82.0,14.0,0,1,0
3,496.0,141.0,20.0,65.0,78.0,37.0,11.0,5628.0,1575.0,225.0,828.0,838.0,354.0,200.0,11.0,3.0,1,0,1
4,321.0,87.0,10.0,39.0,42.0,30.0,2.0,396.0,101.0,12.0,48.0,46.0,33.0,805.0,40.0,4.0,1,0,1
5,594.0,169.0,4.0,74.0,51.0,35.0,11.0,4408.0,1133.0,19.0,501.0,336.0,194.0,282.0,421.0,25.0,0,1,0


In [5]:
knn_model = KNeighborsRegressor().fit(X_train, y_train)

In [6]:
#default komsuluk degeri

In [7]:
knn_model.n_neighbors

5

## Tahmin

In [8]:
y_pred = knn_model.predict(X_test)

In [9]:
#rmse

In [10]:
np.sqrt(mean_squared_error(y_test, y_pred))

np.float64(426.6570764525201)

*Küçük bir gözlem yapalım. Her bir k komşuluk sayısına göre rmse hesabı yapalım. Bunun için döngü kullanacağız.*

In [11]:
RMSE = []

for k in range(10):
    k += 1
    knn_model = KNeighborsRegressor(n_neighbors = k).fit(X_train, y_train)
    y_pred = knn_model.predict(X_train)
    rmse = np.sqrt(mean_squared_error(y_train, y_pred))
    RMSE.append(rmse)
    print("k =", k ,"için RMSE değeri =" , rmse)

k = 1 için RMSE değeri = 0.0
k = 2 için RMSE değeri = 179.52761335480352
k = 3 için RMSE değeri = 205.20157172291863
k = 4 için RMSE değeri = 220.5139794876305
k = 5 için RMSE değeri = 239.6467132541376
k = 6 için RMSE değeri = 243.5904190007242
k = 7 için RMSE değeri = 258.1478781634636
k = 8 için RMSE değeri = 266.05374203349805
k = 9 için RMSE değeri = 269.73782093553376
k = 10 için RMSE değeri = 271.2798300436963


*Bu rmse hesabı ilkel olarak yapılan bir hesap. CV yok, validasyon yok. Çok güvenilir bir rmse hesabı değil. Tahmin bölümünde daha istikrarlı bir parametre belirleme işlemi yapacağız.*

## Model Tuning

In [12]:
knn_params = {'n_neighbors' : np.arange(1,30,1)}

In [13]:
knn = KNeighborsRegressor()

In [14]:
knn_cv_model = GridSearchCV(knn, knn_params, cv = 10)

In [15]:
knn_cv_model.fit(X_train, y_train)

GridSearchCV(cv=10, estimator=KNeighborsRegressor(),
             param_grid={'n_neighbors': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29])})

In [16]:
# En uygun parametre

In [17]:
knn_cv_model.best_params_['n_neighbors']

np.int64(8)

*Peki burada yapmış olduğumuz cv yöntemi ile rmse hesabının yukarıdaki for döngüsü ile yapılan hesaptan ne farkı var ?*

In [18]:
RMSE = []
RMSE_CV = []

for k in range(10):
    k += 1
    knn_model = KNeighborsRegressor(n_neighbors = k).fit(X_train, y_train)
    y_pred = knn_model.predict(X_train)
    rmse = np.sqrt(mean_squared_error(y_train, y_pred))
    rmse_cv = np.sqrt(-1 * cross_val_score(knn_model, X_train, y_train, cv = 10,
                                           scoring = "neg_mean_squared_error")).mean()
    RMSE.append(rmse)
    RMSE_CV.append(rmse)
    print("k =", k ,"için RMSE değeri =" , rmse, "RMSE_CV değeri = ", rmse_cv)

k = 1 için RMSE değeri = 0.0 RMSE_CV değeri =  314.00484847632146
k = 2 için RMSE değeri = 179.52761335480352 RMSE_CV değeri =  286.3603325067769
k = 3 için RMSE değeri = 205.20157172291863 RMSE_CV değeri =  273.9931666607254
k = 4 için RMSE değeri = 220.5139794876305 RMSE_CV değeri =  275.7180399577597
k = 5 için RMSE değeri = 239.6467132541376 RMSE_CV değeri =  278.63565154224034
k = 6 için RMSE değeri = 243.5904190007242 RMSE_CV değeri =  284.4231965885963
k = 7 için RMSE değeri = 258.1478781634636 RMSE_CV değeri =  280.0247076065351
k = 8 için RMSE değeri = 266.05374203349805 RMSE_CV değeri =  276.2836061510732
k = 9 için RMSE değeri = 269.73782093553376 RMSE_CV değeri =  280.327057055243
k = 10 için RMSE değeri = 271.2798300436963 RMSE_CV değeri =  287.23959512744636


*Hemen yukarıdaki for döngüsünde; hem RMSE hem de Cross Validation ile RMSE işlemlerinin çıktıları var. Soldaki değerler direkt olarak train seti üzerinden yapılan hesaplamalar. Sağdaki değerler ise valide edilmiş model üzerinden elde edilmiş ve hepsi train üzerinden elde edilen hatalar. Soldaki değerler arasındaki fark daha fazla yani standart sapması yüksek. Sağdaki değerlerin standart sapması daha düşük yani en düşük değer ile en yüksek değer arasında daha az bir fark var, sapma miktarı daha az. Bu da daha istikrarlı sonuçlar demek. Test hatalarını **valide edilmiş modeller** üzerinden değerlendirmek daha sağlıklıdır.*

In [19]:
#final model

In [20]:
knn_tuned = KNeighborsRegressor(n_neighbors = knn_cv_model.best_params_['n_neighbors'])

In [21]:
knn_tuned.fit(X_train, y_train)

KNeighborsRegressor(n_neighbors=np.int64(8))

In [22]:
#test hatasi

In [23]:
np.sqrt(mean_squared_error(y_test, knn_tuned.predict(X_test)))

np.float64(413.7094731463598)